🐍 Aufgabe: Asynchroner Datei-Downloader

Entwickle ein Python-Programm, das den Download mehrerer Dateien simuliert. Ziel ist es, dabei Generatoren und asyncio miteinander zu kombinieren.

Ausgangssituation

Du hast fünf Dateien:

report.pdf
image.png
data.csv
presentation.pptx
notes.txt
Anforderungen

1. Generator

Erstelle einen Generator, der die vorhandenen Dateien nacheinander liefert.

Die gesamte Liste soll also nicht direkt verarbeitet werden. Stattdessen soll der Generator bei jedem Aufruf die jeweils nächste Datei bereitstellen.

2. Asynchroner Download

Implementiere eine asynchrone Funktion, die den Download einer einzelnen Datei simuliert.

Dabei soll:

der Start des Downloads ausgegeben werden,
der Download zwischen 1 und 3 Sekunden dauern,
während dieser Zeit das Programm nicht blockieren,
anschließend das Ende des Downloads ausgegeben werden.

Die tatsächliche Dauer darf für jede Datei unterschiedlich sein.

3. Mehrere Downloads gleichzeitig

Implementiere eine weitere asynchrone Funktion, die den Generator verwendet, um die Dateien abzurufen.

Für jede vom Generator gelieferte Datei soll ein neuer asynchroner Download gestartet werden.

Die Downloads sollen nebenläufig stattfinden.

Das Programm sollte beispielsweise ungefähr folgende Ausgabe erzeugen:

Downloading report.pdf...
Downloading image.png...
Downloading data.csv...
Downloading presentation.pptx...
Downloading notes.txt...

Finished data.csv
Finished report.pdf
Finished notes.txt
Finished image.png
Finished presentation.pptx


Die Reihenfolge, in der die Downloads abgeschlossen werden, darf variieren.

4. Programm starten

Sorge dafür, dass das asynchrone Programm korrekt aus deinem normalen Python-Programm heraus gestartet wird.

⭐ Bonus 1: Async Generator

Wenn die Grundversion funktioniert, ersetze den normalen Generator durch einen asynchronen Generator.

Simuliere dabei, dass das Ermitteln der nächsten verfügbaren Datei selbst etwa 0,5 Sekunden benötigt.

Passe die Verarbeitung entsprechend an.

⭐⭐ Bonus 2: Maximale Parallelität

Stelle anschließend sicher, dass maximal zwei Dateien gleichzeitig heruntergeladen werden können.

Wenn fünf Downloads anstehen, müssen die übrigen warten, bis einer der beiden laufenden Downloads abgeschlossen ist.

Was du danach beherrschen solltest

Nach der Aufgabe solltest du den Unterschied zwischen diesen vier Konzepten erklären können:

yield → Generator
await → auf eine asynchrone Operation warten
asyncio.create_task() → Coroutine nebenläufig starten
async for → über einen asynchronen Iterator/Generator iterieren

Mein Vorschlag: Löse zuerst nur Punkte 1 bis 4. Die beiden Bonusaufgaben machen anschließend den Unterschied zwischen Generator und Async Generator besonders gut sichtbar.

In [38]:
import time


FILES = ["report.pdf", 
         "image.png",
         "data.csv",
         "presentation.pptx",
         "notes.txt"]

origin = time.time()

# Classic file loop (out of focus)


In [12]:
def handle_files_classic(files):
    for file in files:
        print(f"Handling file: {file}")
        
handle_files_classic(FILES)

Handling file: report.pdf
Handling file: image.png
Handling file: data.csv
Handling file: presentation.pptx
Handling file: notes.txt


# Generator loop (not async)

In [13]:
# Generator function. Yields result to calling function
def handle_files_generator(files):
    for file in files:
        # This is not printing the result
        yield file

# Call generator as an iterator and print the result
for file in handle_files_generator(FILES):
    print(file)


report.pdf
image.png
data.csv
presentation.pptx
notes.txt


# Async download simulation

In [51]:
import time
import random
import asyncio

#origin = time.time()

async def download_file(file):
    #print(f"origin: {origin:.2f}")
    start = time.time()
    #print(f"start: {start:.2f}")
    print(f"Start download of {file}: {start - origin:.2f}")
    #await asyncio.sleep(random.randrange(1, 4))
    await asyncio.sleep(random.randrange(5, 8))
    duration = time.time() - start
    #print(f"duration: {duration:.2f}")
    print(f"Download from {file} completed after {duration:.2f} seconds")

await download_file("test.txt")


Start download of test.txt: 95.96
Download from test.txt completed after 6.01 seconds


# Multiple downloads (async but step by step. No time saved)

In [15]:
async def download_multiple_files():
    # Call generator as an iterator and print the result
    for file in handle_files_generator(FILES):
        await download_file(file)
        
await download_multiple_files()

Start download of report.pdf: 1793982563.458161
Download from report.pdf completed after 3.002971649169922 seconds
Start download of image.png: 1793982566.4612427
Download from image.png completed after 3.0026402473449707 seconds
Start download of data.csv: 1793982569.463988
Download from data.csv completed after 3.0029120445251465 seconds
Start download of presentation.pptx: 1793982572.467051
Download from presentation.pptx completed after 2.0017714500427246 seconds
Start download of notes.txt: 1793982574.4689085
Download from notes.txt completed after 3.002912759780884 seconds


# Multiple and parallel download (async and generator)

- `handle_files_generator(files)` returns a generator that yields filenames one at a time; it does not create a list of filenames.
- `for file in ...` iterates through those filenames.
- `download_file(file)` creates a coroutine for that file. Because it is an async function, calling it creates the coroutine but does not run it yet.
- The generator expression `(download_file(file) for file in ...)` produces those coroutines as it is iterated.
- `*` unpacks the produced coroutines into separate positional arguments. This matters because asyncio.gather expects individual awaitables, not one generator object.
- `asyncio.gather(...)` schedules the coroutines so they can make progress concurrently. When one reaches an await, such as `asyncio.sleep(...)`, another download can proceed.
- `await` waits until all the downloads finish. The gathered results are returned in the same order as the input files, even though the downloads may finish in a different order.

Without *, gather would receive the generator itself rather than the coroutines it yields. This line uses the regular generator version; with your async generator, you need async for to retrieve filenames from it.

The `*` operator unpacks iterables (e.g. a list of arguments) to separate arguments (TypeError if nb of arguments is not matching to the function).
Also possible: print(*my_list) -> prints all items in a row.

In [16]:
async def download_multiple_files_parallel(files):
    await asyncio.gather(*(download_file(file) for file in handle_files_generator(files)))
        
await download_multiple_files_parallel(FILES)

#asyncio.run(download_multiple_files_parallel(FILES))

Start download of report.pdf: 1793982577.4783354
Start download of image.png: 1793982577.4783957
Start download of data.csv: 1793982577.478409
Start download of presentation.pptx: 1793982577.478418
Start download of notes.txt: 1793982577.4784265
Download from data.csv completed after 1.000458002090454 seconds
Download from presentation.pptx completed after 2.0014195442199707 seconds
Download from report.pdf completed after 3.0015063285827637 seconds
Download from image.png completed after 3.0015242099761963 seconds
Download from notes.txt completed after 3.00150465965271 seconds


# Bonus 1 (async generator)

In [17]:
# Generator function. Yields result to calling function
async def handle_files_generator_async(files):
    for file in files:
        await asyncio.sleep(0.5)
        # This is not printing the result
        yield file

# Call generator as an iterator and print the result
#async for file in handle_files_generator_async(FILES):
#    print(file)

In [18]:
async def download_multiple_files_async(file):
    # Call generator as an iterator and print the result
    async for file in handle_files_generator_async(file):
        await download_file(file)
        
await download_multiple_files_async(FILES)

Start download of report.pdf: 1793982580.9947646
Download from report.pdf completed after 2.0020599365234375 seconds
Start download of image.png: 1793982583.4975245
Download from image.png completed after 3.002311944961548 seconds
Start download of data.csv: 1793982587.0005286
Download from data.csv completed after 1.0003092288970947 seconds
Start download of presentation.pptx: 1793982588.5015912
Download from presentation.pptx completed after 1.0002310276031494 seconds
Start download of notes.txt: 1793982590.0025573
Download from notes.txt completed after 2.0013365745544434 seconds


In [19]:
async def download_multiple_files_parallel_async(files):
    tasks = []

    async for file in handle_files_generator_async(files):
        tasks.append(asyncio.create_task(download_file(file)))

    await asyncio.gather(*tasks)

await download_multiple_files_parallel_async(FILES)

#asyncio.run(download_multiple_files_parallel_async(FILES))

Start download of report.pdf: 1793982592.5104775
Start download of image.png: 1793982593.0108764
Download from report.pdf completed after 1.0010778903961182 seconds
Start download of data.csv: 1793982593.5116754
Start download of presentation.pptx: 1793982594.0118878
Start download of notes.txt: 1793982594.512661
Download from presentation.pptx completed after 1.0014986991882324 seconds
Download from image.png completed after 3.000959634780884 seconds
Download from data.csv completed after 3.0010838508605957 seconds
Download from notes.txt completed after 3.001210927963257 seconds


# Bonus 2

In [55]:
sem = asyncio.Semaphore(2)

# Generator function. Yields result to calling function
async def handle_files_generator_async(files):
    for file in files:
        async with sem:
            await asyncio.sleep(0.5)
            # This is not printing the result
            yield file

# Call generator as an iterator and print the result
#async for file in handle_files_generator_async(FILES):
#    print(file)

In [56]:
async def download_multiple_files_parallel_async(files):
    tasks = []

    async for file in handle_files_generator_async(files):
        tasks.append(asyncio.create_task(download_file(file)))

    await asyncio.gather(*tasks)

origin = time.time()
await download_multiple_files_parallel_async(FILES)

#asyncio.run(download_multiple_files_parallel_async(FILES))

Start download of report.pdf: 0.50
Start download of image.png: 1.00
Start download of data.csv: 1.50
Start download of presentation.pptx: 2.00
Start download of notes.txt: 2.50
Download from report.pdf completed after 5.00 seconds
Download from image.png completed after 6.00 seconds
Download from presentation.pptx completed after 5.00 seconds
Download from data.csv completed after 6.00 seconds
Download from notes.txt completed after 5.00 seconds
